# 00 — Exploration des données (Partie 1.2)
**Responsable : DIOP Alioune** · relecture : FALL Madjiguène

`train.csv` est exploré en détail. Pour `test.csv`, on ne lit que les colonnes `subject` et `Activity`
(comptages) : aucune statistique des variables d'entrée du test n'est utilisée pour décider du prétraitement.

In [ ]:
import sys; sys.path.append("../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from har import config, data
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid", font_scale=0.9)
config.FIG_DIR.mkdir(parents=True, exist_ok=True)

## 1. Taille des fichiers, variables et volontaires

In [ ]:
df = data.load_train()
df_test_info = pd.read_csv(config.TEST_CSV, usecols=[config.SUBJECT_COL, config.TARGET_COL])
feats = data.feature_columns(df)

summary = pd.DataFrame({
    "observations": [len(df), len(df_test_info)],
    "variables d'entrée": [len(feats), len(feats)],
    "volontaires": [df[config.SUBJECT_COL].nunique(), df_test_info[config.SUBJECT_COL].nunique()],
}, index=["train", "test"])
display(summary)

s_tr, s_te = set(df[config.SUBJECT_COL]), set(df_test_info[config.SUBJECT_COL])
print("volontaires train :", sorted(s_tr))
print("volontaires test  :", sorted(s_te))
print("volontaires communs :", s_tr & s_te or "aucun → évaluation sur des personnes nouvelles")

## 2. Qualité des données : types, valeurs manquantes, doublons

In [ ]:
print("types :", df[feats].dtypes.value_counts().to_dict())
print("valeurs manquantes :", int(df.isna().sum().sum()))
print("lignes dupliquées :", int(df.duplicated().sum()))
dup_cols = df[feats].T.duplicated()
print("colonnes strictement identiques à une autre :", int(dup_cols.sum()))
const_cols = [c for c in feats if df[c].nunique() <= 1]
print("colonnes constantes :", len(const_cols))

## 3. Distribution des classes

In [ ]:
dist = pd.concat({
    "train (n)": df[config.TARGET_COL].value_counts(),
    "train (%)": df[config.TARGET_COL].value_counts(normalize=True).mul(100).round(1),
    "test (n)": df_test_info[config.TARGET_COL].value_counts(),
    "test (%)": df_test_info[config.TARGET_COL].value_counts(normalize=True).mul(100).round(1),
}, axis=1).reindex(config.CLASSES)
display(dist)
print("rapport classe max / classe min (train) :",
      round(dist["train (n)"].max() / dist["train (n)"].min(), 2))

fig, ax = plt.subplots(figsize=(9, 4))
dist[["train (%)", "test (%)"]].plot.bar(ax=ax)
ax.set_ylabel("% des observations"); ax.set_title("Distribution des classes"); plt.xticks(rotation=30, ha="right")
plt.tight_layout(); plt.savefig(config.FIG_DIR / "p1_class_distribution.png", dpi=200)

In [ ]:
# nombre de fenêtres par volontaire et par activité (train)
per_subj = pd.crosstab(df[config.SUBJECT_COL], df[config.TARGET_COL])[config.CLASSES]
fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(per_subj, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
ax.set_title("Fenêtres par volontaire et par activité (train)")
plt.tight_layout(); plt.savefig(config.FIG_DIR / "p1_windows_per_subject.png", dpi=200)
print("fenêtres par volontaire : min", per_subj.sum(axis=1).min(), "max", per_subj.sum(axis=1).max())

## 4. Statistiques descriptives
Les variables sont-elles déjà mises à l'échelle ?

In [ ]:
desc = df[feats].describe().T[["mean", "std", "min", "max"]]
display(desc.head(15))
print(f"min global = {desc['min'].min():.3f} · max global = {desc['max'].max():.3f}")
in_range = ((desc["min"] >= -1 - 1e-9) & (desc["max"] <= 1 + 1e-9)).mean()
print(f"part des variables dans [-1, 1] : {in_range:.1%}")
print(f"moyennes : de {desc['mean'].min():.3f} à {desc['mean'].max():.3f}")
print(f"écarts-types : de {desc['std'].min():.3f} à {desc['std'].max():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(desc["mean"], bins=50); axes[0].set_title("Moyennes des 561 variables")
axes[1].hist(desc["std"], bins=50);  axes[1].set_title("Écarts-types des 561 variables")
plt.tight_layout(); plt.savefig(config.FIG_DIR / "p1_means_stds.png", dpi=200)

In [ ]:
# familles de variables (préfixe t = domaine temporel, f = fréquentiel ; capteur ; statistique)
fam = pd.Series(feats).str.extract(r"^(?P<domaine>[tf]|angle)?(?P<signal>[A-Za-z]+)?")
print("domaine :", fam["domaine"].fillna("autre").value_counts().to_dict())

## 5. Visualisations : histogrammes et boxplots par activité

In [ ]:
show = [c for c in ["tBodyAccMag-mean()", "tGravityAcc-mean()-X", "fBodyAccMag-mean()",
                    "angle(X,gravityMean)", "tBodyAcc-std()-X", "tBodyGyro-std()-X"] if c in df.columns]
if len(show) < 4:                      # au cas où les noms diffèrent
    show = list(desc["std"].sort_values(ascending=False).index[:6])

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, c in zip(axes.ravel(), show):
    for k in config.CLASSES:
        ax.hist(df.loc[df[config.TARGET_COL] == k, c], bins=40, alpha=0.5, label=k, density=True)
    ax.set_title(c)
axes[0, 0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(config.FIG_DIR / "p1_histograms.png", dpi=200)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, c in zip(axes.ravel(), show):
    sns.boxplot(data=df, x=config.TARGET_COL, y=c, order=config.CLASSES, ax=ax, fliersize=1)
    ax.set_xlabel(""); ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.savefig(config.FIG_DIR / "p1_boxplots.png", dpi=200)

## 6. Corrélations

In [ ]:
corr = df[feats].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
pairs = upper.stack()
for t in (0.9, 0.95, 0.99):
    print(f"paires de variables avec |r| > {t} : {(pairs > t).sum()} sur {len(pairs)}")

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(df[feats[:60]].corr(), cmap="coolwarm", center=0, ax=ax, xticklabels=False, yticklabels=False)
ax.set_title("Corrélations entre les 60 premières variables")
plt.tight_layout(); plt.savefig(config.FIG_DIR / "p1_correlations.png", dpi=200)

In [ ]:
# corrélation de chaque variable avec la cible (rapport de corrélation η² : part de variance expliquée par l'activité)
def eta2(x, y):
    g = x.groupby(y)
    return (g.count() * (g.mean() - x.mean()) ** 2).sum() / ((x - x.mean()) ** 2).sum()
eta = pd.Series({c: eta2(df[c], df[config.TARGET_COL]) for c in feats}).sort_values(ascending=False)
display(eta.head(10).to_frame("η² avec Activity"))

## 7. La colonne `subject` doit-elle servir d'entrée ?

In [ ]:
# subject comme variable : quelle part de variance des entrées est liée au volontaire (à activité fixée) ?
resid = df[feats] - df.groupby(config.TARGET_COL)[feats].transform("mean")
subj_share = (resid.groupby(df[config.SUBJECT_COL]).transform("mean").var() / resid.var()).median()
print(f"part médiane de la variance intra-activité expliquée par le volontaire : {subj_share:.1%}")

## 8. Discussion : besoins de prétraitement

*À rédiger à partir des sorties ci-dessus (points à couvrir) :*

- **Mise à l'échelle** : les variables sont déjà normalisées dans [-1, 1] par les auteurs du jeu de données,
  mais leurs moyennes et écarts-types restent très différents → on applique quand même un `StandardScaler`
  **ajusté sur l'apprentissage seulement**, ce qui aide la descente de gradient. Comparer les chiffres obtenus.
- **Encodage** : `Activity` → entiers 0–5 (`data.encode_labels`) ; pas de one-hot, `CrossEntropyLoss` attend des indices.
- **Suppression de colonnes** : pas de valeurs manquantes ; noter le nombre de colonnes dupliquées / très corrélées.
  On les garde pour le MLP (les poids gèrent la redondance), à discuter.
- **`subject`** : **non** utilisée en entrée. C'est un identifiant, pas une mesure ; les volontaires du test sont
  absents du train, donc le modèle ne pourrait rien en tirer et risquerait d'apprendre des particularités de personnes.
  Elle sert en revanche à construire la validation (Partie 1.3).
- **Classes** : distribution assez équilibrée (rapport max/min ci-dessus) → l'accuracy reste lisible, mais le F1 macro
  et par classe sont plus informatifs pour les paires difficiles (SITTING / STANDING).